## Simple MNE example

In [1]:
import numpy as np
import mne
from mne.datasets import sample
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score



In [4]:
from pathlib import Path
from mne.datasets import sample
import mne

data_path = sample.data_path()

raw = mne.io.read_raw_fif(
    data_path / "MEG" / "sample" / "sample_audvis_raw.fif",
    preload=True
)

Opening raw data file C:\Users\Neermita\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...


In [5]:
raw

<Raw | sample_audvis_raw.fif, 376 x 166800 (277.7 s), ~481.7 MiB, data loaded>

In [6]:
# Select channels of interest (EEG channels)
picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)

In [7]:
picks

array([315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327,
       328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340,
       341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353,
       354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366,
       368, 369, 370, 371, 372, 373, 374])

In [8]:
# Set the events and event_id
events = mne.find_events(raw, stim_channel='STI 014')
event_id = {'left/auditory': 1, 'right/auditory': 2}

Finding events on: STI 014
320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]


In [9]:
# Create epochs around events
epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.5, picks=picks, baseline=(None, 0), preload=True)
labels = epochs.events[:, -1]

# Extract data and labels
X = epochs.get_data()
y = labels

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Not setting metadata
145 matching events found
Setting baseline interval to [-0.19979521315838786, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 145 events and 421 original time points ...
0 bad epochs dropped


In [10]:
labels

array([2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1,
       2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 2, 1, 2,
       1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1,
       2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2,
       1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 2, 1, 2,
       1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2, 1, 2, 1, 2, 1, 2, 1, 2, 2,
       1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1])

In [16]:
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

# Initialize a classifier
svm = SVC(kernel='linear', C=1)

# Create a pipeline
clf = Pipeline([('CSP', csp), ('SVM', svm)])

# Train the classifier
clf.fit(X_train, y_train)

# Predict the labels for the test set
y_pred = clf.predict(X_test)

# Evaluate the classifier
print("Classification report:\n", classification_report(y_test, y_pred))
print("Accuracy score:", accuracy_score(y_test, y_pred))

Computing rank from data with rank=None


    Using tolerance 0.00029 (2.2e-16 eps * 59 dim * 2.2e+10  max singular value)
    Estimated rank (data): 59
    data: rank 59 computed from 59 data channels with 0 projectors
Reducing data rank from 59 -> 59
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Classification report:
               precision    recall  f1-score   support

           1       0.42      0.33      0.37        15
           2       0.41      0.50      0.45        14

    accuracy                           0.41        29
   macro avg       0.41      0.42      0.41        29
weighted avg       0.41      0.41      0.41        29

Accuracy score: 0.41379310344827586


## My data

In [1]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from mne.decoding import CSP
from sklearn.decomposition import PCA


In [4]:
# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
subject = "LTP063"
session = "0"
base_path = r"C:\Users\Neermita\Desktop\memory_and_task\ds004395"

sub_dir = f"sub-{subject}"
ses_dir = f"ses-{session}"

edf_path = os.path.join(base_path, sub_dir, ses_dir, "eeg", f"{sub_dir}_{ses_dir}_task-ltpFR_eeg.edf")
events_path = os.path.join(base_path, sub_dir, ses_dir, "eeg", f"{sub_dir}_{ses_dir}_task-ltpFR_events.tsv")
elec_path = os.path.join(base_path, sub_dir, ses_dir, "eeg", f"{sub_dir}_{ses_dir}_space-CapTrak_electrodes.tsv")

raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


In [5]:
raw

<RawEDF | sub-LTP063_ses-0_task-ltpFR_eeg.edf, 129 x 2603000 (5206.0 s), ~101 KiB, data not loaded>

In [6]:
# Montage Setup
electrodes = pd.read_csv(elec_path, sep='\t')
electrodes = electrodes[electrodes['x'] != 'n/a'].copy()
ch_pos = {row['name']: [float(row['x']), float(row['y']), float(row['z'])] 
          for _, row in electrodes.iterrows() if row['name'] in raw.ch_names}
raw.set_montage(mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head'), on_missing='ignore')

<RawEDF | sub-LTP063_ses-0_task-ltpFR_eeg.edf, 129 x 2603000 (5206.0 s), ~145 KiB, data not loaded>

In [5]:
import pandas as pd
import numpy as np
df= pd.read_csv(r"C:\Users\Neermita\Desktop\memory_and_task\code\results\ers_metrics.csv")

In [6]:
df

,subject,session,n_class0,n_class1,n_balanced,acc,acc_std,theta_class0,alpha_class0,theta_class1,alpha_class1,n_words,n_recalls,raw_recall_rate,correct_recall_rate
0,LTP063,0,23,5,5,0.700000,0.244949,NaN,NaN,NaN,NaN,256,114,0.445312,0.425781
1,LTP063,1,241,120,120,0.875000,0.047507,2.026127e-12,9.169479e-13,4.712047e-12,1.300575e-12,256,137,0.535156,0.511719
2,LTP063,2,47,24,24,0.726667,0.108321,NaN,NaN,NaN,NaN,256,153,0.597656,0.574219
3,LTP063,3,33,14,14,0.640000,0.110353,NaN,NaN,3.076207e-12,6.829214e-13,256,159,0.621094,0.609375
4,LTP063,4,33,19,19,0.925000,0.100000,NaN,NaN,NaN,NaN,256,194,0.757812,0.742188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,LTP067,15,254,323,254,0.722520,0.050139,1.512924e-12,8.704856e-13,3.192668e-12,1.202152e-12,256,344,1.343750,0.679688
95,LTP067,16,255,260,255,0.764706,0.036683,1.055087e-12,6.213908e-13,2.804009e-12,1.039750e-12,256,260,1.015625,0.589844
96,LTP067,17,248,224,224,0.738851,0.021583,1.663054e-12,1.093071e-12,3.925232e-12,1.625672e-12,256,245,0.957031,0.644531
97,LTP067,18,249,234,234,0.707390,0.058365,1.460466e-12,1.109400e-12,3.087122e-12,1.108733e-12,256,239,0.933594,0.589844


In [12]:
max(df["acc"])

0.975189393939394

In [ ]:
import os
import mne
import pandas as pd
import matplotlib.pyplot as plt

# 1. Point to just one session to test (e.g., LTP063, Session 0)
base_path = r"C:\Users\Neermita\Desktop\memory_and_task\ds004395\sub-LTP063\ses-0\eeg"
eeg_file = os.path.join(base_path, "sub-LTP063_ses-0_task-ltpFR_eeg.edf")
electrode_file = os.path.join(base_path, "sub-LTP063_ses-0_space-CapTrak_electrodes.tsv")

# 2. Load the raw data (preload=False makes this instant)
raw = mne.io.read_raw_edf(eeg_file, preload=False, verbose=False)

# 3. Parse the exact 3D coordinates from the TSV file
electrodes = pd.read_csv(electrode_file, sep="\t")
valid_electrodes = electrodes[electrodes["x"].notna() & (electrodes["x"] != "n/a")]

ch_pos = {
    row["name"]: [float(row["x"]), float(row["y"]), float(row["z"])]
    for _, row in valid_electrodes.iterrows()
    if row["name"] in raw.ch_names
}

# 4. Apply the montage to the raw object
montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame="head")
raw.set_montage(montage, on_missing="ignore")

# 5. Plot the 2D top-down view with all names visible!
fig = raw.plot_sensors(
    show_names=True, 
    title="PEERS Dataset Electrode Montage"
)

# Enlarge the figure for better readability
fig.set_size_inches(12, 12)
plt.show()

ValueError: sphere="eeglab" requires digitization points of the following electrode locations in the data: Fpz, Oz, T7, T8, but could not find: Fpz, and was unable to approximate its location from Oz